# SAM3 一本勝負 - ボール・選手・ゴール・コート全検出
**ランタイム → T4 GPU を選択してから実行**

In [ ]:
# セル1: GPU確認
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# セル2: インストール
!pip install -q transformers>=5.5.0 opencv-python-headless

In [ ]:
# セル3: HuggingFace ログイン
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
print('ログイン完了')

In [ ]:
# セル4: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# セル5: SAM3 モデルロード
from transformers import Sam3Model, Sam3Processor
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'デバイス: {DEVICE}')

print('SAM3 読み込み中...')
processor = Sam3Processor.from_pretrained('facebook/sam3')
model = Sam3Model.from_pretrained('facebook/sam3', torch_dtype=torch.float16).to(DEVICE)
model.eval()
print('SAM3 ロード完了')

In [ ]:
# セル6: 検出関数
import cv2
import numpy as np
from PIL import Image

# 検出対象とその設定
TARGETS = {
    'ball':    {'prompt': 'basketball',         'conf': 0.25, 'max_area': 8000,   'single': True},
    'player':  {'prompt': 'basketball player',  'conf': 0.20, 'max_area': 200000, 'single': False},
    'hoop':    {'prompt': 'basketball hoop',    'conf': 0.20, 'max_area': 50000,  'single': True},
    'court':   {'prompt': 'basketball court',   'conf': 0.20, 'max_area': None,   'single': True},
}

def mask_to_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

def mask_to_centroid(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return [float(xs.mean()), float(ys.mean())]

def infer_objects(frame_bgr):
    """1フレームで全オブジェクト検出"""
    H, W = frame_bgr.shape[:2]
    img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    result = {}

    for key, cfg in TARGETS.items():
        inputs = processor(images=img, text=cfg['prompt'], return_tensors='pt')
        inputs = {k: v.to(DEVICE).half() if v.dtype == torch.float32 else v.to(DEVICE)
                  for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        try:
            res = processor.post_process_instance_segmentation(
                outputs, threshold=cfg['conf'], mask_threshold=0.5,
                target_sizes=[(H, W)]
            )[0]
        except Exception:
            result[key] = None if cfg['single'] else []
            continue

        masks = res['masks'].cpu().numpy().astype(bool)
        scores = res['scores'].cpu().numpy()

        # サイズフィルタ
        valid = []
        for i, mask in enumerate(masks):
            area = mask.sum()
            if area < 100:
                continue
            if cfg['max_area'] and area > cfg['max_area']:
                continue
            valid.append((scores[i], mask))

        if not valid:
            result[key] = None if cfg['single'] else []
            continue

        if cfg['single']:
            best_mask = max(valid, key=lambda x: x[0])[1]
            result[key] = mask_to_centroid(best_mask)
        else:
            # 選手: 上位8人まで、bboxも保存
            valid.sort(key=lambda x: -x[0])
            players = []
            for score, mask in valid[:8]:
                c = mask_to_centroid(mask)
                b = mask_to_bbox(mask)
                if c and b:
                    players.append({'centroid': c, 'bbox': b, 'score': float(score)})
            result[key] = players

    return result

print('検出関数定義完了')
print('検出対象:', list(TARGETS.keys()))

In [ ]:
# セル7: 1フレームテスト
import matplotlib.pyplot as plt

VIDEO_PATH = '/content/drive/MyDrive/basketball_analysis/game_EE1swQMsXJc_720p.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 300)  # 10秒目
ret, test_frame = cap.read()
cap.release()

print('1フレームテスト中...')
result = infer_objects(test_frame)
print('\n=== 検出結果 ===')
print(f'ボール  : {result["ball"]}')
print(f'ゴール  : {result["hoop"]}')
print(f'コート  : {result["court"]}')
print(f'選手数  : {len(result["player"])}人')
for i, p in enumerate(result['player']):
    print(f'  選手{i+1}: centroid={[round(v) for v in p["centroid"]]}  score={p["score"]:.2f}')

# 可視化
vis = test_frame.copy()
if result['ball']:
    bx, by = int(result['ball'][0]), int(result['ball'][1])
    cv2.circle(vis, (bx, by), 15, (0, 140, 255), -1)
    cv2.putText(vis, 'BALL', (bx+15, by), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,140,255), 2)
if result['hoop']:
    hx, hy = int(result['hoop'][0]), int(result['hoop'][1])
    cv2.circle(vis, (hx, hy), 20, (0, 255, 0), 3)
    cv2.putText(vis, 'HOOP', (hx+20, hy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
for i, p in enumerate(result['player']):
    x1,y1,x2,y2 = p['bbox']
    cv2.rectangle(vis, (x1,y1), (x2,y2), (255,200,0), 2)
    cv2.putText(vis, f'P{i+1}', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,200,0), 2)

plt.figure(figsize=(14,8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('SAM3 One-Model Detection Test (t=10s)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('/content/sam3_test_frame.png', dpi=100, bbox_inches='tight')
plt.show()
print('テスト完了')

In [ ]:
# セル8: 60秒分を処理（FRAME_SKIPフレームおきに推論）
import json
from tqdm import tqdm

SEC        = 60
FRAME_SKIP = 5   # 5フレームおきに推論（約6fps相当）

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(SEC * fps)
print(f'処理フレーム数: {n_frames} ({SEC}秒)  推論: {n_frames//FRAME_SKIP}回')

all_results = {}  # {frame_idx: {ball, player, hoop, court}}
prev = None

for fi in tqdm(range(n_frames), desc='SAM3 全オブジェクト検出'):
    ret, frame = cap.read()
    if not ret:
        break

    if fi % FRAME_SKIP == 0:
        result = infer_objects(frame)
        prev = result
    else:
        result = prev  # スキップフレームは前の結果を流用

    # JSON化できる形式に変換
    all_results[fi] = {
        'ball':   result['ball'] if result else None,
        'hoop':   result['hoop'] if result else None,
        'court':  result['court'] if result else None,
        'players': result['player'] if result else [],
    }

cap.release()

# 統計
n_ball   = sum(1 for v in all_results.values() if v['ball'])
n_hoop   = sum(1 for v in all_results.values() if v['hoop'])
n_player = sum(1 for v in all_results.values() if v['players'])
print(f'\n=== 検出結果 ===')
print(f'ボール  : {n_ball}/{n_frames} ({100*n_ball/n_frames:.1f}%)')
print(f'ゴール  : {n_hoop}/{n_frames} ({100*n_hoop/n_frames:.1f}%)')
print(f'選手あり: {n_player}/{n_frames} ({100*n_player/n_frames:.1f}%)')

In [ ]:
# セル9: 保存してダウンロード
from google.colab import files

out_path = '/content/sam3_all_objects_60s.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f)

size_kb = len(open(out_path).read()) / 1024
print(f'保存完了: {size_kb:.0f} KB')
files.download(out_path)
print('ダウンロード開始！')

## ダウンロード後
```bash
# ローカルで可視化
python sam3_all_visualize.py --json sam3_all_objects_60s.json
```